# Build 04-01 · SHAP-DiD inputs — region (A/B) x era (early/late) tags, per (version, split)

### 실행 전 설정

**커널**: analysis `.venv` (`python3`). **바꿀 설정 없음** — v1/v2/v3 전부 이 노트북 안에서
자동으로 만듭니다. (v1은 2026-09-16부터 포함 — `score > 0.85` 고정 임계값 fallback 사용, §1
주석 참고.)

**선행 조건**: v1/v2/v3의 `processed_inputs`/`targets`/`scores`(base export)가 train과 그
버전의 OOT split(v1=`val2`, v2=`test`, v3=`oot`)에 대해 이미 있어야 합니다 — 이건 이
파이프라인의 가장 기초 데이터라 보통은 이미 있습니다.

In [ ]:
# §0 — setup (analysis .venv kernel)
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import schema
import figstyle
import feature_alias
from loaders import load

figstyle.apply()
figstyle.FIG_DIR = ROOT / "figures" / "shap_did" / "04_01"
figstyle.ALIAS_SPLIT = True  # real-name/alias top-feature outputs -> real_named/ / alias/
pd.set_option("display.width", 160)
print("ROOT =", ROOT)
print("FIG_DIR =", figstyle.FIG_DIR)

In [ ]:
# §1 — SOURCES / RUN_SPEC
ID_COL = schema.CLAIM_ID

# v1 IS built here (2026-09-16), for the headline v1->v2 pair `04_02` already computes --
# proving parallel trends breaks at the v1->v2 era jump needs v1's OWN region tags, not just
# v2/v3's. v1 has no scalar tau (DECISION_RULES["v1"] is SEGMENTED on mobility, which is
# unrecoverable -- see project_v1_mobility_not_a_feature / DATA_MODEL.md SS0.1b), so region
# here reuses the SAME flat `score > 0.85` fallback `loaders.VersionData.decisions` already
# applies for v1 everywhere else in this codebase (00_SHAP_v1.ipynb's TAU, 04_02's own
# `region_tag_inline` fallback before this change) -- not a new approximation, the existing
# one, now PERSISTED instead of recomputed inline. Caveat carried forward unchanged: this
# undercounts immobile scraps in the (0.75, 0.85] overlap band (false negatives only, never
# false positives) -- see DATA_MODEL.md SS0.1b for the full error profile.
#
# v1 -> v2 still breaks parallel trends from the era jump ALONE (human-handler labels
# 2016-01 to 2018-02 -> model-log labels), independent of this threshold choice -- v2 -> v3
# remains the only pair read as a substantive estimate; v1 -> v2 stays a placebo/validity
# check (04_02 SS2's markdown). Persisting v1's tags here does not change that reading, it
# only lets 04_02 show the SAME diagnostic against a comparable, on-disk artefact.
#
# Both TRAIN (long window, more rows to halve into early/late) and OOT (short, may be too thin to
# sub-split -- check §5's cell counts before trusting it) are built, mirroring
# notebook/real/mitigation/03_01_corrector_inputs.ipynb's BUILD pattern.
BUILD = {v: list(dict.fromkeys(["train", config.OOT_SPLIT[v]])) for v in ("v1", "v2", "v3")}

print("build plan:", BUILD)

In [ ]:
# §2 — helpers: era split (early/late) + region (A/B, strict >) + the write step
def assign_era(dates: pd.Series) -> tuple[pd.Series, pd.Timestamp]:
    """early/late by the MEDIAN date within this population. A PLACEHOLDER split.

    Median split is the safe default because it works identically for v2 (piecewise tau, five
    regimes) and v3 (single global tau, never deployed) without hand-picking a version-specific
    cutoff. A real per-version regime-aware split (e.g. v2's own tau-regime breaks,
    config.DECISION_RULES["v2"]["regimes"]) may be a sharper choice once real dates are visible —
    see Notes at the bottom of this notebook.
    """
    dates = pd.to_datetime(dates)
    cutoff = dates.median()
    era = pd.Series(np.where(dates <= cutoff, "early", "late"), index=dates.index)
    return era, cutoff


def build_shap_did_input(version: str, split: str) -> Path:
    """targets + scores + that version's OWN decision rule -> region (A/B) + era (early/late).

    Deliberately does NOT touch the attributions parquet (the phi matrix) — this is a small
    claim-level tag table. 04-02 joins it onto attributions by claim_id and computes the
    concentration measures (Hill / Shannon / Simpson) per (region, era) cell from there.
    """
    d = load(version, split=split)
    df = d.frame.copy()
    df[schema.DECISION] = d.decisions
    df["region"] = np.where(df[schema.DECISION] == 1, "B", "A")  # B: score > tau, STRICT
    df["era"], cutoff = assign_era(df[schema.DATE])

    out = df[[ID_COL, schema.DATE, d.score_col, schema.DECISION, "region", "era",
              schema.OBSERVED]].rename(columns={d.score_col: "score"})

    p = config.split_path("shap_did_input", version, split)
    p.parent.mkdir(parents=True, exist_ok=True)
    out.to_parquet(p, index=False)

    counts = out.groupby(["era", "region"])[ID_COL].count().to_dict()
    p.with_name(p.stem + "_meta.json").write_text(json.dumps({
        "version": version, "split": split, "n_claims": int(len(out)),
        "era_cutoff_date": str(cutoff.date()),
        "era_rule": "median date within this (version, split) population — PROVISIONAL",
        "region_rule": "decision==1 -> B (score>tau, strict); else A",
        "cell_counts": {f"{k[0]}/{k[1]}": int(v) for k, v in counts.items()},
        "columns": list(out.columns),
    }, indent=2), encoding="utf-8")

    print(f"[{version}] {split}: {len(out):,} claims | era cutoff {cutoff.date()} | cells {counts}")
    print(f"  -> {p.name}")
    return p

In [ ]:
# §2b — v1: build shap_did_input for every split in BUILD (flat score > 0.85 fallback -- see §1)
for split in BUILD["v1"]:
    build_shap_did_input("v1", split)

In [ ]:
# §3 — v2: build shap_did_input for every split in BUILD
for split in BUILD["v2"]:
    build_shap_did_input("v2", split)

In [ ]:
# §4 — v3: build shap_did_input for every split in BUILD
for split in BUILD["v3"]:
    build_shap_did_input("v3", split)

In [ ]:
# §5 — what exists now
rows = []
for v, sps in BUILD.items():
    for sp in sps:
        p = config.split_path("shap_did_input", v, sp)
        if p.is_file():
            meta = json.loads(p.with_name(p.stem + "_meta.json").read_text(encoding="utf-8"))
            rows.append({"version": v, "split": sp, "n_claims": meta["n_claims"],
                         "era_cutoff": meta["era_cutoff_date"], **meta["cell_counts"]})
        else:
            rows.append({"version": v, "split": sp, "n_claims": "(not built)"})
coverage = pd.DataFrame(rows).set_index(["version", "split"])
display(coverage)
# Index is (version, split), not a feature name -- plain export, no alias twin needed.
figstyle.save_table(coverage, "05_shap_did_input_coverage")

In [ ]:
# §6 — sanity plot: mean|SHAP| by feature — TWO INDEPENDENT figures, real name and alias
# (proves the anonymisation wiring works, matching notebook/real/00_shap_attribution.ipynb's
# convention: the real-name PNG is company-laptop/internal only, the alias_ one is the only
# version safe to leave the machine). Each is its own plt.subplots()/tight_layout()/savefig() —
# not one figure relabelled and saved twice, so the alias figure's layout is never sized around
# the (usually longer) real-name labels. The backing table gets the same real/alias split.
# Needs that (version, split)'s attributions parquet to already exist (built by
# attribution/attribute.py, run once per version, or notebook/real/00_SHAP.ipynb) — this notebook
# does not build it.
def _bar_chart(mabs: pd.Series, labels: list[str], title: str, fname: str) -> None:
    fig, ax = plt.subplots(figsize=(6.5, 0.4 * len(mabs) + 1.2))
    bars = ax.barh(np.arange(len(mabs)), mabs.values)
    ax.set_yticks(np.arange(len(mabs)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("mean |SHAP|")
    ax.set_title(title)
    ax.bar_label(bars, fmt="%.3f", fontsize=6, padding=2)
    ax.margins(x=0.12)     # headroom so the value label never clips past the axes edge
    fig.tight_layout()
    figstyle.save_fig(fig, fname)
    plt.show()


def plot_top_features_aliased(version: str, split: str, k: int = 10) -> None:
    d = load(version, split=split)
    mabs = d.mean_abs_shap.sort_values(ascending=False).head(k)[::-1]     # ascending, for barh
    real_labels = list(mabs.index)
    alias_labels = feature_alias.to_alias(version, real_labels)

    _bar_chart(mabs, real_labels, f"{version} . {split} . top {k} features",
               f"{version}_{split}_06a_shap_did_top_features")
    _bar_chart(mabs, alias_labels, f"{version} . {split} . top {k} features (aliased)",
               f"alias_{version}_{split}_06a_shap_did_top_features")

    # TWO INDEPENDENT tables backing the two figures above — same real/alias split, descending
    # (the figures are ascending only because barh draws bottom-up).
    table = pd.DataFrame({"feature": real_labels[::-1], "mean_abs_shap": mabs.values[::-1]}
                          ).set_index("feature")
    figstyle.save_table(table, f"{version}_{split}_06a_shap_did_top_features")
    alias_table = table.set_axis(alias_labels[::-1], axis=0).rename_axis("feature")
    figstyle.save_table(alias_table, f"alias_{version}_{split}_06a_shap_did_top_features")


for v, sps in BUILD.items():
    try:
        plot_top_features_aliased(v, sps[0])
    except FileNotFoundError as exc:
        print(f"[{v}] skip sanity plot (attributions not built yet) — {exc}")

## Notes

- **v1 is now built too (2026-09-16), with a flat threshold fallback, not v1's real segmented
  rule.** v1's decision rule is SEGMENTED on mobility (`0.75` immobile / `0.85` mobile), and
  mobility is unrecoverable from either the raw dataset or a production log (v1 has none) — see
  `DATA_MODEL.md` §0.1b. Region here reuses the SAME `score > 0.85` fallback
  `loaders.VersionData.decisions` already applies for v1 everywhere else in this codebase, so
  this is the existing approximation, persisted, not a new one. **v1 -> v2 still breaks parallel
  trends from the era jump alone** (human-handler labels, 2016-01 to 2018-02, -> model-log
  labels), independent of forced-label dose or this threshold choice — `04_02`'s
  `cross_version_estimate("v1", ...)` therefore stays a placebo/validity check, never a
  substantive estimate; v2 -> v3 remains the one DiD contrast read that way.
- **Region is strict `>`.** `score > tau` -> region B (scrapped); `score <= tau` -> region A.
  Never `>=` -- see `src/threshold.py` and the project's threshold convention.
- **Era is a MEDIAN-date placeholder, not a confirmed break.** Revisit once real dates are visible
  on the company laptop. `config.DECISION_RULES["v2"]["regimes"]` is one candidate for a sharper,
  version-aware split; v1/v3 have no equivalent (v1's rule has no date axis at all; v3 is a
  single global tau, never deployed), so whatever rule is chosen must still make sense for every
  side of every contrast.
- **Feature names are never printed raw without an alias twin.** Any figure OR table with feature
  names on an axis, legend, or index must go through `feature_alias.to_alias(version, names)`
  before it is saved to disk. §6 above builds TWO INDEPENDENT figures per (version, split) --
  `_bar_chart()` is called once on the real feature names
  (`<version>_<split>_06a_shap_did_top_features.png`, company-laptop / internal use only) and once
  on the aliased names (`alias_<version>_<split>_06a_shap_did_top_features.png` -- the only
  version safe to leave the machine), each with its own `plt.subplots()`/`tight_layout()`/
  `savefig()` rather than one figure relabelled and saved twice, so neither figure's layout is
  sized around the other's label widths. The backing table gets the identical treatment --
  `<version>_<split>_06a_shap_did_top_features.csv` (real) and
  `alias_<version>_<split>_06a_shap_did_top_features.csv` (aliased), same feature-index rule.
  Same convention as `notebook/real/00_shap_attribution.ipynb`'s §1/§7 figures. The mapping itself
  -- `features/registry/feature_alias_map.json` -- is built once on the company laptop by
  `features/build_feature_alias.py` and never reaches git (already covered by
  `features/registry/*.json` in `.gitignore`). §0 sets `figstyle.ALIAS_SPLIT = True` (2026-09-16)
  alongside `figstyle.FIG_DIR = figures/shap_did/04_01/`, so the real-name PNGs/CSVs land under
  `figures/shap_did/04_01/real_named/` and the safe-to-share ones under `.../alias/` automatically
  -- `figures/` is NOT gitignored, so only ever stage the `alias/` subfolder, never `real_named/`.
- **This notebook writes TAGS only** (`shap_did_input`, claim_id grain: region, era, score, date,
  observed). The concentration measures (Hill / Shannon / Simpson) per (region, era) cell, and the
  `(B-A)|late - (B-A)|early` subtraction itself, are 04-02's job -- it joins this output onto the
  attributions parquet by `claim_id`.